# RASA mit LLM

Ziele der heutigen Übung:
- RASA an ein LLM anbinden (CALM)
- Zwei Flows (Öffnungszeiten und Pizza Bestellen) mit RASA CALM implementieren

Wir haben bereits RASA in der NLU-Variante kennen gelernt. In der neuen Variante von RASA werden LLMs zum Verstehen der Nutzeräußerung, Slot-Mapping und zur Antwortgenrierung genutzt. 

Lese dir das Konzept dahinter auf folgender Seite durch: https://rasa.com/docs/learn/concepts/calm 

Wir erstellen zuerst ein RASA Projekt mit folgenden Eigenschaften:
1) ```rasa init``` aufrufen
2) als Installationspfad folgendes angeben: ```rasa_calm```
3) Create Path => Y
4) CALM (default) auswählen
5) Train => n

Nun haben wir ein initiales Projekt, das bereits eine einfache Anwendung in Form eines Telefonbuchs beinhaltet. Bei der folgenden eigenen Implementierung hilft es ungemein, wenn man sich die entsprechenden Files wie add_contact, list_contacts und remove_contact.yml als Beispiel nimmt.

RASA CALM nutzt ein LLM für die NLU und NLG. Wir nutzen den fakultätsinternen LLM-Server, um Anfragen mit einem "gpt-oss:20b" Modell zu verarbeiten (VPN notwendig). Wer einen Computer mit entsprechendem RAM und Grafikkarte hat kann auch gerne ein lokales LLM in https://lmstudio.ai/ laufen lassen und dagegen implementieren. Über URL, API_KEY und MODEL_NAME konfiguriert folgendes Skript die endpoints.yml Datei und legt eine Datei mit Umgebungsvariablen für API_KEY und RASA_PRO_LICENSE an.

*Anmerkung: den API_KEY findet ihr im Moodle Kurs, als RASA_PRO_LICENSE eure eigene Lizenz nutzen*

In [ ]:
URL = "http://141.75.33.244:8000/v1/"
API_KEY = "<hier den API KEY aus Moodle einfügen>"
MODEL_NAME = "gpt-oss:20b"
RASA_PRO_LICENSE = "<hier den eigenen RASA Lizenz Key einfügen>"

new_group = {
    "id": "dgx_spark_ME",
    "models": [
        {
            "provider": "openai",
            "model": "gpt-oss:20b",
            "api_base": "http://141.75.33.244:8000/v1/",
            "max_tokens": 4096,
            "timeout": 2000
        }
    ],
}

from ruamel.yaml import YAML
import os

yaml_io = YAML()
yaml_io.preserve_quotes = True

with open(os.path.join("rasa_calm", "endpoints.yml"), "r", encoding="utf-8") as f:
    data = yaml_io.load(f)

groups = data.setdefault("model_groups", [])

existing = next((g for g in groups if g.get("id") == new_group['id']), None)

if existing is not None:
    for key, value in new_group.items():
        existing[key] = value
    print(f"Model group '{new_group['id']}' aktualisiert.")
else:
    groups.append(new_group)
    print(f"Model group '{new_group['id']}' neu angelegt.")

with open(os.path.join("rasa_calm", "endpoints.yml"), "w", encoding="utf-8") as f:
    yaml_io.dump(data, f)

##### .env File erzeugen
with open(os.path.join("rasa_calm", ".env"), "w", encoding="utf-8") as f:
    f.write(f"RASA_PRO_LICENSE={RASA_PRO_LICENSE}\n")
    f.write(f"OPENAI_API_KEY={API_KEY}\n")

Model group 'dgx_spark_ME' aktualisiert.


Schau dir die generierten Dateien ```.env``` und ```endpoints.yml``` an. 

Nun müssen wir unsere ```config.yml``` noch etwas umschreiben, so dass der LLMCommandGenerator den Endpunkt dgx_spark_ME nutzt. Flow_retrieval schalten wir aus, da wir sonst ein zusätzliches Embedding Model benötigen. Bei größeren Dialogsystemen bietet sich dies an, da dies die Token-Anzahl bei der LLM-Anfrage verringert (siehe Vorlesung LLM und RAG).

```
- name: CompactLLMCommandGenerator
  llm:
    model_group: dgx_spark_ME
  flow_retrieval:
    active: false
```

Nun können wir die bekannten Kommandos ```rasa train``` und ```rasa shell``` nutzen, um unseren Assistenten auszuführen. 

Probiert auch mal ```rasa inspect``` aus, das startet eine ganz nette UI im Browser.

### Implementieren der beiden Flows

Wie mit der NLU-Variante wollen wir folgende beiden Flows implementieren (am besten in separaten .yml Files):
- opening_hours:

    a) allgemein für alle Wochentage

    b) an einem spezifischen Wochentag

- order_pizza:

    a) Pizza Belag, z.B. Salami, Vegetarisch, Thunfisch, etc.

    b) Anzahl der Pizzen

    c) extra Käse

Folgendes Video gibt dir einen Überblick wie man Flows in RASA CALM nutzt:
https://www.youtube.com/watch?v=v1l75qxo4b0&list=PL75e0qA87dlHPWoD4c-NrYszndgq-NFz3&index=5

Anschließend noch für jede Flow-Datei eine Domain-Datei anlegen, in welchen ihr Responses und Slots definiert. Damit sollte man bereits einen Assistenten schaffen der sehr brauchbar funktioniert :).